# GLM-OCR: base vs LoRA on CoSyn chemical molecules

Runs the whole experiment on a Colab GPU: baseline eval, LoRA training, fine-tuned eval, summary.

**Before running:** Runtime → Change runtime type → **T4 GPU**.

## 1. Setup

In [3]:
!git clone https://github.com/MahnoorAhmed-Dev/glm-ocr-lora-cosyn-chem.git
%cd glm-ocr-lora-cosyn-chem
!pip install -q -e LLaMA-Factory rdkit
# transformers needs huggingface-hub 1.x; do not upgrade it to 2.x
!pip install -q "huggingface_hub>=1.5,<2.0"
!hf download zai-org/GLM-OCR --local-dir GLM-OCR

Cloning into 'glm-ocr-lora-cosyn-chem'...
remote: Enumerating objects: 569, done.
remote: Counting objects: 100% (569/569), done.
remote: Compressing objects: 100% (498/498), done.
remote: Total 569 (delta 68), reused 559 (delta 61), pack-reused 0 (from 0)
Receiving objects: 100% (569/569), 7.46 MiB | 24.56 MiB/s, done.
Resolving deltas: 100% (68/68), done.
/content/glm-ocr-lora-cosyn-chem/glm-ocr-lora-cosyn-chem
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Installing backend dependencies ... done
  Preparing editable metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 846.4/846.4 kB 33.3 MB/s eta 0:00:00
  Building editable for llamafactory (pyproject.toml) ... done
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
hf-gradio 0.4

In [4]:
import torch, transformers
print(torch.__version__, transformers.__version__, torch.cuda.get_device_name(0))

2.11.0+cu130 5.8.0 Tesla T4


## 2. Baseline: untouched GLM-OCR on the held-out set (30 molecules)

In [5]:
!python scripts/evaluate.py     --data LLaMA-Factory/data/cosyn_chem_eval.json --data-root LLaMA-Factory/data     --model GLM-OCR --out results/base.json

[transformers] Missing validation function in 'RotaryEmbeddingConfigMixin' for 'rope_type'='axial'
[transformers] Missing validation function in 'RotaryEmbeddingConfigMixin' for 'rope_type'='axial'
Loading weights: 100% 510/510 [00:00<00:00, 16616.14it/s]
[1/30] match=False sim=0.4286  CCCC(=O)C(O)C
[2/30] match=True sim=1.0  CCOC=O
[3/30] match=False sim=0.2807  Cc1cc(C)c(O)c(C)c1OcC(O)c1cc(C)c(O)c1OcC(O)c1cc(C)c(O)c1C
[4/30] match=False sim=0.1721  C/C=C/C(C)=C/C=C/C(C)=C/C=C/C(C)=C/C=C/C(C)=C/C=C/C(C)=C/C=C
[5/30] match=False sim=0.1139  $$
\mathrm {C u} ^ {+} \mathrm {N} ^ {-} \mathrm {N} ^ {+} \
[6/30] match=False sim=0.183  CCCCCCCCCCCCCCCCCCCCC(=O)C(=O)COC(=O)COC(=O)COC(=O)COC(=O)CO
[7/30] match=False sim=0.129  C(C)C(C)C(C)C(C)C(C)C(C)C(C)C(C)C(C)C(C)C(C)C(C)C(C)C(C)C(C)
[8/30] match=False sim=0.3  C(=O)C(=C)C(=O)[C@@H](O)CO
[9/30] match=True sim=1.0  CC=O
[10/30] match=False sim=0.25  C(N)CCC(=O)O<sup>+</sup>C[C@@H](C(=O)O)N<sup>+</sup>
[11/30] match=False sim=0.9667  C(C)(C)C

## 3. LoRA fine-tune (200 training molecules, 3 epochs)

In [6]:
%cd LLaMA-Factory
!llamafactory-cli train data/glm_ocr_lora_colab.yaml
%cd ..

/content/glm-ocr-lora-cosyn-chem/glm-ocr-lora-cosyn-chem/LLaMA-Factory
/usr/local/lib/python3.13/dist-packages/jieba/__init__.py:44: SyntaxWarning: invalid escape sequence '\.'
  re_han_default = re.compile("([\u4E00-\u9FD5a-zA-Z0-9+#&\._%\-]+)", re.U)
/usr/local/lib/python3.13/dist-packages/jieba/__init__.py:46: SyntaxWarning: invalid escape sequence '\s'
  re_skip_default = re.compile("(\r\n|\s)", re.U)
/usr/local/lib/python3.13/dist-packages/jieba/finalseg/__init__.py:78: SyntaxWarning: invalid escape sequence '\.'
  re_skip = re.compile("([a-zA-Z0-9]+(?:\.\d+)?%?)")
/usr/local/lib/python3.13/dist-packages/jieba/_compat.py:18: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead

## 4. Fine-tuned: base + LoRA adapter on the same held-out set

In [7]:
!python scripts/evaluate.py     --data LLaMA-Factory/data/cosyn_chem_eval.json --data-root LLaMA-Factory/data     --model GLM-OCR --adapter LLaMA-Factory/saves/glm-ocr/lora/colab --out results/lora.json

[transformers] Missing validation function in 'RotaryEmbeddingConfigMixin' for 'rope_type'='axial'
[transformers] Missing validation function in 'RotaryEmbeddingConfigMixin' for 'rope_type'='axial'
Loading weights: 100% 510/510 [00:00<00:00, 6787.52it/s]
[1/30] match=False sim=0.7143  CCCCC(=O)O
[2/30] match=True sim=1.0  CCOC=O
[3/30] match=False sim=0.6429  CC1=CC(=C(C=C1C=C/C(=C2C(=C(C=C2C=C1)O)C)O)OC)C(O)CO
[4/30] match=False sim=0.5534  CC1CCCC(C)(C)C(=C1C)/C=C/C=C/C(C)=C/C=C/C=C/C(C)=C/C=C/C(C)=
[5/30] match=False sim=0.35  [N+](=N=N=N)Cu
[6/30] match=False sim=0.8852  CCCCCCCCCCCCCCC(=O)OC(COC(=O)CCCCCCCCCCCC)OC(=O)CCCCCCCCCCCC
[7/30] match=False sim=0.1587  ```rdkit
CC(C#N)C(C)C=CC(C#N)CC(C#N)(C)C(C#N)C(C#N)C(C#N)C(C
[8/30] match=False sim=0.3667  O=C1C(=C(C(=O)O1)[C@H](CO)O)O
[9/30] match=True sim=1.0  CC=O
[10/30] match=False sim=0.4242  CN(CCC(=O)O)C(=O)C[C@H](C(=O)O)N
[11/30] match=True sim=1.0  CC(C)(C)C(=O)OCCOC(=O)C(C)(C)C
[12/30] match=False sim=0.4  O=Cc1ccc(O)cc1O
[13

## 5. Summary and download

In [8]:
!python scripts/make_summary.py
from IPython.display import Markdown
Markdown(open("results/summary.md").read())

# Results: base GLM-OCR vs LoRA fine-tuned

Held-out evaluation on 30 CoSyn chemical molecules that were never used for training.

| Metric | Base | LoRA | Change |
|---|---|---|---|
| Exact match (canonical SMILES) | 6.7% | 23.3% | +16.7 pts |
| Valid SMILES | 50.0% | 70.0% | +20.0 pts |
| Mean character similarity | 31.8% | 61.1% | +29.3 pts |



# Results: base GLM-OCR vs LoRA fine-tuned

Held-out evaluation on 30 CoSyn chemical molecules that were never used for training.

| Metric | Base | LoRA | Change |
|---|---|---|---|
| Exact match (canonical SMILES) | 6.7% | 23.3% | +16.7 pts |
| Valid SMILES | 50.0% | 70.0% | +20.0 pts |
| Mean character similarity | 31.8% | 61.1% | +29.3 pts |

## Per-sample outputs

| # | Target | Base | LoRA |
|---|---|---|---|
| 1 | `CCO.CCCCC(=O)O` | ❌ `CCCC(=O)C(O)C` | ❌ `CCCCC(=O)O` |
| 2 | `CCOC=O` | ✅ `CCOC=O` | ✅ `CCOC=O` |
| 3 | `CC1=CC(=C(C(=C1O)C)OC)C(C=CC2=CC(=C(C(=C2O)C)OC)C(CO)O)O` | ❌ `Cc1cc(C)c(O)c(C)c1OcC(O)c1cc(C)c(O)c1OcC(O)c1cc(C)c(O)c1C` | ❌ `CC1=CC(=C(C=C1C=C/C(=C2C(=C(C=C2C=C1)O)C)O)OC)C(O)CO` |
| 4 | `CC1=C(C(CCC1)(C)C)C=CC(=CC=CC(=CC=CC=C(C)C=CC=C(C)C=CC2=C(CCCC2(C)C)C)C)C` | ❌ `C/C=C/C(C)=C/C=C/C(C)=C/C=C/C(C)=C/C=C/C(C)=C/C=C/C(C)=C/C=C/C(C)=C/C=C/C(C)=C/C=C/C(C)=C/C=C/C(C)=C/C=C/C(C)=C/C=C/C(C)=C/C=C/C(C)=C/C=C/C(C)=C/C=C/C(C)=C/C=C/C(C)=C/C=C/C(C)=C/C=C/C(C)=C/C=C/C(C)=C/C=C/C(C)=C/C=C/C(C)=C/C=C/C(C)=C/C=C/C(C)=C/C=C/C(C)=C/C=C/C(C)=C/C=C/C(C)=C/C=C/C(C)=C/C=C/C(C)=C/C=C/C(C)=C/C=C/C(C)=C/C=C/C(C)=C/C=C/C(C)=C/C=C/C(C)=C/C=C/C(C)=C/C=C/C(C)=C/C=C/C(C)=C/C=C/C(C)=C/C=C` | ❌ `CC1CCCC(C)(C)C(=C1C)/C=C/C=C/C(C)=C/C=C/C=C/C(C)=C/C=C/C(C)=C/C=C/C(C)(C)C)/C=C/C(C)(C)=C/CCCC1=C(C)CCC` |
| 5 | `[Cu+].[N-]=[N+]=[N-]` | ❌ `$$
\mathrm {C u} ^ {+} \mathrm {N} ^ {-} \mathrm {N} ^ {+} \mathrm {N} ^ {-}
$$` | ❌ `[N+](=N=N=N)Cu` |
| 6 | `CCCCCCCCCCCC(=O)OCC(COC(=O)CCCCCCCCCCC)OC(=O)CCCCCCCCCCC` | ❌ `CCCCCCCCCCCCCCCCCCCCC(=O)C(=O)COC(=O)COC(=O)COC(=O)COC(=O)COC(=O)COC(=O)COC(=O)COC(=O)COC(=O)COC(=O)COC(=O)COC(=O)COC(=O)COC(=O)COC(=O)COC(=O)COC(=O)COC(=O)COC(=O)COC(=O)COC(=O)COC(=O)COC(=O)COC(=O)COC(=O)COC(=O)COC(=O)COC(=O)COC(=O)COC(=O)COC(=O)COC(=O)COC(=O)COC(=O)COC(=O)COC(=O)COC(=O)COC(=O)COC(=O)COC` | ❌ `CCCCCCCCCCCCCCC(=O)OC(COC(=O)CCCCCCCCCCCC)OC(=O)CCCCCCCCCCCCC` |
| 7 | `CC(C)(C#N)CC(C)C=CC(c1ccccc1)CC(C)(C#N)CC(C)C=CC(c2ccccc2)CC(C)(C#N)` | ❌ `C(C)C(C)C(C)C(C)C(C)C(C)C(C)C(C)C(C)C(C)C(C)C(C)C(C)C(C)C(C)C(C)C(C)C(C)C(C)C(C)C(C)C(C)C(C)C(C)C(C)C(C)C(C)C(C)C(C)C(C)C(C)C(C)C(C)C(C)C(C)C(C)C(C)C(C)C(C)C(C)C(C)C(C)C(C)C(C)C(C)C(C)C(C)C(C)C(C)C(C)C(C)C(C)C(C)C(C)C(C)C(C)C(C)C(C)C(C)C(C)C(C)C(C)C(C)C(C)C(C)C(C)C(C)C(C)C(C)C(C)C(C)C(C)C(C)C(C)C(C)C(C)C(C)C(C)C(C)C(C)C(C)C(C)C(C)C(C)C(C)C` | ❌ ````rdkit
CC(C#N)C(C)C=CC(C#N)CC(C#N)(C)C(C#N)C(C#N)C(C#N)C(C#N)C(C#N)C(C#N)C(C#N)C(C#N)C(C#N)C(C#N)C(C#N)C(C#N)C(C#N)C(C#N)C(C#N)C(C#N)C(C#N)C(C#N)C(C#N)C(C#N)C(C#N)C(C#N)C(C#N)C(C#N)C(C#N)C(C#N)C(C#N)C(C#N)C(C#N)C(C#N)C(C#N)C(C#N)C(C#N)C(C#N)C(C#N)C(C#N)C(C#N)C(C#N)C(C#N)C(C#N)C(C#N)C(C#N)C(C#N)C(C#N)C(C#N)C(C#N)` |
| 8 | `O=C1O[C@H]([C@H](CO)O)C(O)=C1O` | ❌ `C(=O)C(=C)C(=O)[C@@H](O)CO` | ❌ `O=C1C(=C(C(=O)O1)[C@H](CO)O)O` |
| 9 | `CC=O` | ✅ `CC=O` | ✅ `CC=O` |
| 10 | `N[C@@H](CC(=O)N(CCC(=O)O)C)C(=O)O` | ❌ `C(N)CCC(=O)O<sup>+</sup>C[C@@H](C(=O)O)N<sup>+</sup>` | ❌ `CN(CCC(=O)O)C(=O)C[C@H](C(=O)O)N` |
| 11 | `CC(C)(C)C(=O)OCCOC(=O)C(C)(C)C` | ❌ `C(C)(C)C(=O)OCCOC(=O)C(C)(C)C` | ✅ `CC(C)(C)C(=O)OCCOC(=O)C(C)(C)C` |
| 12 | `Oc1ccc(/C=C/c2cc(O)cc(O)c2)cc1` | ❌ `C=Cc1cc(O)cc(O)c1<sep>OH</sep>` | ❌ `O=Cc1ccc(O)cc1O` |
| 13 | `O=Cc1ccccc1` | ❌ `C=OC1=CC=C1` | ✅ `O=Cc1ccccc1` |
| 14 | `CC12CCC(=O)C=C1CCC3C2CCC4(C(=O)CCC34C)C(=O)CO` | ❌ `C(C)(C)C1(C)C2C3C4C(C)(C)C(C(=O)C)C(C)(C)C4C3C2C1C3C4C(C)(C)C(C(=O)C)C4C3C2C1C3C4C(C)(C)C(C(=O)C)C4C3C2C1C3C4C(C)(C)C(C(=O)C)C4C3C2C1C3C4C(C)(C)C(C(=O)C)C4C3C2C1C3C4C(C)(C)C(C(=O)C)C4C3C2C1C3C4C(C)(C)C(C(=O)C)C4C3C2C1C3C4C(C)(C)C(C(=O)C)C4C3C2C1C3C4C(C)(C)C(C(=O)C)C4C3C2C1C3C4C(C)(C)C(C(=O)` | ❌ `CC1(CCC2C3CCC(C2C1=CC(=O)CC3)(C)C)C(=O)CCO` |
| 15 | `[Na+].[Na+].[O-]S(=O)(=O)OS(=O)(=O)[O-]` | ❌ `S(=O)(=O)(=O)S(=O)(=O)O(S(=O)(=O)(=O)S(=O)(=O)(=O)O)` | ❌ `[Na+].[Na+].[O-].[O-].[S-].[S-].[O-].[O-].[O-].[O-]` |
| 16 | `C1=NC(=C2C(=N1)N(C=N2)C3C(C(C(O3)COP(=O)(O)OP(=O)(O)OP(=O)(O)O)O)O)N` | ❌ `C(C(O)C(O)C(O)C(O)C(O)C(O)C(O)C(O)C(O)C(O)C(O)C(O)C(O)C(O)C(O)C(O)C(O)C(O)C(O)C(O)C(O)C(O)C(O)C(O)C(O)C(O)C(O)C(O)C(O)C(O)C(O)C(O)C(O)C(O)C(O)C(O)C(O)C(O)C(O)C(O)C(O)C(O)C(O)C(O)C(O)C(O)C(O)C(O)C(O)C(O)C(O)C(O)C(O)C(O)C(O)C(O)C(O)C(O)C(O)C(O)C(O)C(O)C(O)C(O)C(O)C(O)C(O)C(O)C(O)C(O)C(O)C(O)C(O)C(O)C(O)C(O)C(O)C(O)C(O)C(O)C(O)C(O)C(O)C(O)C(O)` | ❌ `Nc1ncnc2c1ncn2C(O)C(O)C(O)C(O)OP(O)(=O)OP(O)(=O)OP(O)(=O)O` |
| 17 | `[Na+].[Cl-].[O-]Cl` | ❌ `$$
\mathrm {O} ^ {-} \mathrm {C l} \mathrm {N a} ^ {+}
$$` | ❌ `[Na+].[O-].[Cl-]` |
| 18 | `CC1=C(C(CCC1)(C)C)C=CC(=C)C=O` | ❌ `C=C(CC=O)C=C/C1=C(CCC1(C)C)C` | ❌ `CC1=C(C(CCC1)(C)C)/C=C/C(=C)C=O` |
| 19 | `C(C1C(C(C(C(O1)OC2(C(C(C(O2)CO)O)O)CO)O)O)O)O` | ❌ `C(C)C(O)C(O)C(O)C(O)C(O)C(O)C(O)C(O)C(O)C(O)C(O)C(O)C(O)C(O)C(O)C(O)C(O)C(O)C(O)C(O)C(O)C(O)C(O)C(O)C(O)C(O)C(O)C(O)C(O)C(O)C(O)C(O)C(O)C(O)C(O)C(O)C(O)C(O)C(O)C(O)C(O)C(O)C(O)C(O)C(O)C(O)C(O)C(O)C(O)C(O)C(O)C(O)C(O)C(O)C(O)C(O)C(O)C(O)C(O)C(O)C(O)C(O)C(O)C(O)C(O)C(O)C(O)C(O)C(O)C(O)C(O)C(O)C(O)C(O)C(O)C(O)C(O)C(O)C(O)C(O)C(O)C(O)C(O)C(O)C` | ❌ `CC1C(OC(OC2C(C1O)CO)OCC2)OCCO` |
| 20 | `COC(=O)C1=CC=CC=C1O` | ❌ `C(O)C(=O)C1=C=C=C1O` | ✅ `COC(=O)c1ccccc1O` |
| 21 | `[Ag+].[Br-]` | ❌ `<html><body><table><tr><td>$Br^{-}$</td><td>$Ag^{+}$</td></tr><tr><td>1</td><td>0</td></tr></table></body></html>` | ❌ `[Br]^- [Ag+]` |
| 22 | `CC12CCC(=O)C=C1CCC3C4CCC(C(=O)CO)(C(O)CC34C)C2` | ❌ `C1C2C3C(C2C1C3C(O)C(=O)CO)C(C)(C)C(C)(C)C(C)(C)C(C)(C)C(C)(C)C(C)(C)C(C)(C)C(C)(C)C(C)(C)C(C)(C)C(C)(C)C(C)(C)C(C)(C)C(C)(C)C(C)(C)C(C)(C)C(C)(C)C(C)(C)C(C)(C)C(C)(C)C(C)(C)C(C)(C)C(C)(C)C(C)(C)C(C)(C)C(C)(C)C(C)(C)C(C)(C)C(C)(C)C(C)(C)C(C)(C)C(C)(C)C(C)(C)C(C)(C)C(C)(C)C(C)(C)C(C)(C)C(C)(C)C(C)(C)C(C)(C)C(C)(C)C(C)(C)C(C)(C)C(C)(C)C(C)(C)C(C)(C)C(C)(` | ❌ `CC12CCC3CC(C12)C(CCC3C(O)C)C(=O)CO` |
| 23 | `CCCCCC` | ❌ `CCCCC` | ❌ `CCCCC` |
| 24 | `O=C1OC(=O)C=C1` | ❌ `C=C C(=O)O C(=O)` | ✅ `O=C1C=CC(=O)O1` |
| 25 | `[Co@OH3]([OH2])([OH2])([OH2])([OH2])([OH2])` | ❌ `$$
\mathrm {C o} \mathrm {O} _ {4} \mathrm {H} _ {2} \mathrm {O} \mathrm {O H} _ {2} \mathrm {O} \mathrm {O H} _ {2}
$$` | ❌ `[Co]([H2O])([OH])([H2O])[OH]O` |
| 26 | `CN(C)c1cccc2cccc(N(C)C)c12` | ❌ `C(N)(C)C(=C)C1=C2C=CC=C2C(N)(C)C1=C2C=CC=C2C(N)(C)C1=C2C=CC=C2C(N)(C)C1=C2C=CC=C2C(N)(C)C1=C2C=CC=C2C(N)(C)C1=C2C=CC=C2C(N)(C)C1=C2C=CC=C2C(N)(C)C1=C2C=CC=C2C(N)(C)C1=C2C=CC=C2C(N)(C)C1=C2C=CC=C2C(N)(C)C1=C2C=CC=C2C(N)(C)C1=C2C=CC=C2C(N)(C)C1=C2C=CC=C2C(N)(C)C1=C2C=CC=C2C(N)(C)C1=C2C=CC=C2C(N)(C)C1=C2C` | ❌ `CN(C)C1=C2C=CC=C2C(=C1)N(C)C` |
| 27 | `NNc1ccccc1` | ❌ `C=C(N)C=C` | ✅ `NNc1ccccc1` |
| 28 | `C1CO1` | ❌ `Chemical structure of a triene with three carbon atoms and one oxygen atom.` | ❌ `O1C2C1=O` |
| 29 | `c1ccc(cc1)O` | ❌ `C(C=C)C=C` | ❌ `Oc1ccc2cccc12` |
| 30 | `CC1CCN2CC(C1)C(C(C2)O)C3=CC=CN4C3=CC=C4` | ❌ `C(C)C1CC2C(C(C(C(C(C2N3C=CC=C3)O)O)O)C=C)C` | ❌ `CC1CC2CN3CC1C(C2C3)C(=CC=C1)C4=CC=CN4` |


In [9]:
# Bundle only the small files worth committing: results + the LoRA adapter (+ loss plot).
!mkdir -p bundle/adapter
!cp results/*.json results/summary.md bundle/
!cp LLaMA-Factory/saves/glm-ocr/lora/colab/{adapter_config.json,adapter_model.safetensors,training_loss.png,trainer_log.jsonl,all_results.json} bundle/adapter/ 2>/dev/null
!cd bundle && zip -rq ../results_bundle.zip . && ls -la ../results_bundle.zip
from google.colab import files
files.download("results_bundle.zip")

-rw-r--r-- 1 root root 13859933 Oct 19 09:00 ../results_bundle.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>